In [35]:
from mezcal_library import CoupledModel, CoupledModelExperiment, CoupledModelLibrary
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


[autoreload of mezcal_library failed: Traceback (most recent call last):
  File "/Users/damrhein/anaconda3/lib/python3.11/site-packages/IPython/extensions/autoreload.py", line 276, in check
    superreload(m, reload, self.old_objects)
  File "/Users/damrhein/anaconda3/lib/python3.11/site-packages/IPython/extensions/autoreload.py", line 475, in superreload
    module = reload(module)
             ^^^^^^^^^^^^^^
  File "/Users/damrhein/anaconda3/lib/python3.11/importlib/__init__.py", line 169, in reload
    _bootstrap._exec(spec, module)
  File "<frozen importlib._bootstrap>", line 621, in _exec
  File "<frozen importlib._bootstrap_external>", line 936, in exec_module
  File "<frozen importlib._bootstrap_external>", line 1074, in get_code
  File "<frozen importlib._bootstrap_external>", line 1004, in source_to_code
  File "<frozen importlib._bootstrap>", line 241, in _call_with_frames_removed
  File "/Users/damrhein/Documents/2024-2025/mezcal/MEZCAL/mezcal_library.py", line 136
    else 

In [3]:
scenario = 'piControl'
running_mean_window = 20

lat_min = 0
lat_max = 90

lon_min = -100
lon_max = 15

PATH_SST = '../data/SST/' + scenario + '/'
PATH_AMOC = '../data/AMOC/' + scenario + '/'
PATH_saved_regressions = '../data/SST_AMOC_saved_regressions/'
PATH_saved_cross_regressions = '../data/SST_AMOC_cross_regressions_for_every_model/'

model_names = [
    'ACCESS-CM2', 'ACCESS-ESM1-5', 'CESM2-WACCM', 'CESM2', 'CMCC-CM2-SR5', 'CMCC-ESM2',
    'CNRM-CM6-1-HR', 'CNRM-CM6-1', 'CNRM-ESM2-1', 'CanESM5-CanOE', 'CanESM5',
    'GFDL-CM4', 'GFDL-ESM4', 'GISS-E2-1-G', 'HadGEM3-GC31-LL', 'HadGEM3-GC31-MM',
    'IPSL-CM6A-LR', 'MIROC6', 'MPI-ESM1-2-HR', 'MPI-ESM1-2-LR', 'MRI-ESM2-0',
    'NorESM2-LM', 'NorESM2-MM', 'UKESM1-0-LL'
]

amoc_institution_names = [
    'CSIRO-ARCCSS', 'CSIRO', 'NCAR', 'NCAR', 'CMCC', 'CMCC', 'CNRM-CERFACS',
    'CNRM-CERFACS', 'CNRM-CERFACS', 'CCCma', 'CCCma', 'NOAA-GFDL', 'NOAA-GFDL', 'NASA-GISS',
    'MOHC', 'MOHC', 'IPSL', 'MIROC', 'DKRZ', 'MPI-M', 'MRI', 'NCC', 'NCC', 'MOHC'
]

ens_members_appendices = [
    'r1i1p1f1', 'r1i1p1f1', 'r1i1p1f1', 'r1i1p1f1', 'r1i1p1f1', 'r1i1p1f1', 'r1i1p1f2',
    'r1i1p1f2', 'r1i1p1f2', 'r1i1p2f1', 'r1i1p1f1', 'r1i1p1f1', 'r1i1p1f1', 'r1i1p1f2',
    'r1i1p1f1', 'r1i1p1f1', 'r1i1p1f1', 'r1i1p1f1', 'r1i1p1f1', 'r1i1p1f1', 'r1i1p1f1',
    'r1i1p1f1', 'r1i1p1f1', 'r1i1p1f2'
]

In [ ]:
# === Construct and register all models ===
library = CoupledModelLibrary()
models = []

for name, inst, member in zip(model_names, amoc_institution_names, ens_members_appendices):
    model = CoupledModel(name, inst, member, scenario)
    try:
        model.load_data(PATH_SST, PATH_AMOC, lat_min=-30, lat_max=30, lon_min=0, lon_max=360)
        model.smooth_data(running_mean_window=11)
        model.train_regression()
        library.add_model(model)
        models.append(model)
    except Exception as e:
        print(f"Skipping model {name} due to error: {e}")

# === Create experiments for all (train, truth) combinations ===
for model_train in models:
    for model_truth in models:
        experiment = CoupledModelExperiment(model_train, model_truth)
        library.add_experiment(experiment)

# === Run all inversions ===
library.run_all_experiments(return_impact_map=True, normalize=True)

# === Example: Print RMSE summary ===
print("\nRMSE summary:")
for (train_name, truth_name), exp in library.experiments.items():
    print(f"Train: {train_name:20s} | Truth: {truth_name:20s} | RMSE: {exp.rmse:.4f}")

  # Leave-one-out cross validation